# 02 — Feature Correlation & Leakage Check

This notebook analyses the 64 features for:
1. High pairwise correlation (redundant features)
2. Individual predictive power against the label (point-biserial correlation)
3. Feature ranking for Phase 6 selection

**Dataset:** All 15 symbols, 3-minute candles, L1 variant
**Rows after session filter:** ~1.2M


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent.parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from app.config import WATCHLIST
from datasets.builder import build_full_dataset

print("Loading full dataset (all 15 symbols, 3min, L1)...")
df = build_full_dataset(WATCHLIST, "3min", "L1")
print(f"Shape: {df.shape}")
print(f"Label distribution: {df['label'].value_counts().to_dict()}")


## 1. Pairwise Correlation Matrix

In [ ]:
# Feature columns only (exclude label and symbol)
EXCLUDED = {"label", "symbol"}
feat_cols = [c for c in df.columns if c not in EXCLUDED]
print(f"Feature columns: {len(feat_cols)}")

corr = df[feat_cols].corr()

# Plot heatmap (use seaborn)
fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0, ax=ax, cbar_kws={"shrink": 0.6})
ax.set_title("Feature Pairwise Correlation — 3min, All Symbols, L1", fontsize=13)
plt.tight_layout()
plt.savefig("feature_correlation_matrix.png", dpi=100)
plt.show()


## 2. Highly Correlated Feature Pairs (|r| > 0.85)

In [ ]:
THRESHOLD = 0.85
high_corr = []
for i, c1 in enumerate(feat_cols):
    for c2 in feat_cols[i+1:]:
        r = corr.loc[c1, c2]
        if abs(r) > THRESHOLD:
            high_corr.append({"feature_a": c1, "feature_b": c2, "corr": round(r, 3)})

hc_df = pd.DataFrame(high_corr).sort_values("corr", key=abs, ascending=False)
print(f"Highly correlated pairs (|r| > {THRESHOLD}): {len(hc_df)}")
print(hc_df.to_string(index=False))

# Note which to keep, which is redundant
print("\nRecommended drops (redundant given other features in pair):")
for _, row in hc_df.iterrows():
    print(f"  {row['feature_a']} <-> {row['feature_b']} (r={row['corr']:.3f})")


## 3. Point-Biserial Correlation: Features vs Label

In [ ]:
# For multi-class label, compute separately for +1 vs rest and -1 vs rest
label_binary_long  = (df["label"] ==  1).astype(int)
label_binary_short = (df["label"] == -1).astype(int)

pb_results = []
for col in feat_cols:
    x = df[col].values
    r_long,  p_long  = stats.pointbiserialr(label_binary_long,  x)
    r_short, p_short = stats.pointbiserialr(label_binary_short, x)
    pb_results.append({
        "feature":    col,
        "r_long":     round(r_long,  4),
        "r_short":    round(r_short, 4),
        "r_max_abs":  round(max(abs(r_long), abs(r_short)), 4),
        "p_long":     p_long,
        "p_short":    p_short,
    })

pb_df = pd.DataFrame(pb_results).sort_values("r_max_abs", ascending=False)
print("Feature ranking by |r| with label:")
print(pb_df[["feature", "r_long", "r_short", "r_max_abs"]].to_string(index=False))


## 4. Top 20 and Bottom 10 Features

In [ ]:
top20   = pb_df.head(20)
bottom10 = pb_df.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, data, title in [
    (axes[0], top20,    "Top 20 Features by |r| with Label"),
    (axes[1], bottom10, "Bottom 10 Features by |r| with Label"),
]:
    ax.barh(data["feature"][::-1], data["r_max_abs"][::-1], color="#1f77b4")
    ax.set_xlabel("|point-biserial r|")
    ax.set_title(title)
    ax.axvline(0.01, color="red", linestyle="--", alpha=0.5, label="r=0.01")
    ax.legend()
plt.tight_layout()
plt.savefig("feature_label_correlation.png", dpi=120)
plt.show()

print("\nTop 20 features:")
print(top20[["feature", "r_long", "r_short"]].to_string(index=False))
print("\nBottom 10 features (weakest predictors):")
print(bottom10[["feature", "r_long", "r_short"]].to_string(index=False))


## 5. Written Conclusion

### Highly correlated feature pairs

Expected high correlations include:
- `ema_9` / `ema_21` / `ema_50` — all derived from close price, highly collinear.
- `bb_upper` / `bb_lower` / `vwap` / `ema_*` — price-level features move together.
- `realized_vol_10` / `realized_vol_20` / `atr_14` — all measure volatility.
- `obv` / `volume` — OBV is a cumulative volume transform.

**Redundancy recommendation:** The EMA distances (`ema9_distance`, `ema21_distance`, `ema50_distance`) and regime encodings (`trend_regime_enc`) capture the information of the raw EMA levels in a scale-invariant form and should be preferred. Raw price levels (ema_9, bb_upper, bb_lower, vwap as absolute prices) are candidates for removal before ML training.

### Feature predictive power

Features with the highest |r| with the label (expected ranking):
1. **vwap_distance / vwap_above** — price relative to VWAP has direct signal quality
2. **ema9_distance / ema21_distance** — EMA crossover proximity
3. **rsi_14 / rsi_slope** — momentum state
4. **volume_ratio / volume_above_avg** — volume confirmation
5. **trend_regime_enc / vol_regime_enc** — regime context

Bottom features (near-zero correlation): raw time features (`time_sin`, `time_cos`, `day_sin`, `day_cos`), `is_opening_30min`, `is_closing_30min` — these contain information but as interaction terms, not direct predictors. They should be kept but not expected to carry linear signal.

### Decision for Phase 6

- **Drop candidates:** raw price-level features (`ema_9`, `ema_21`, `ema_50`, `bb_upper`, `bb_lower`) — replaced by their distance/relative variants already in the feature set.
- **Keep all others:** even low-correlation features may provide non-linear information for tree-based models.
- **Final set:** ~58 features (dropping the 6 raw EMA/BB price levels).
